# vitalPeriodic

"Periodic" data refers to data which is consistently interfaced from bedside vital signs monitors into eCareManager. Data are generally interfaced as 1 minute averages, and archived into the vitalPeriodic table as 5 minute median values.

The vital signs are **not** validated by care staff, so data quality can vary, but the use of 1 minute averages followed by 5 minute medians before archival removes many spurious readings.

In [ ]:
from pathlib import Path

import holoviews as hv
import hvplot.pandas  # noqa+
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from loguru import logger

from notebooks_dani.eicu_db_connection import DatabaseConnection
from datetime import datetime

import palettable


hv.extension("bokeh")

%matplotlib inline

In [ ]:
entire_dataset = False # plot save and file flag

In [ ]:
today = datetime.today().strftime('%Y-%m-%d')
notebook_name = "vitalperiodic_outlier_removal"

plot_outputs_dir_small = "subset"
plot_outputs_dir_all = "all"

INTER_DATA_PATH = Path("..") / Path("..") / Path("intermediary_data")
logger.info(f"Loading from {Path(INTER_DATA_PATH).resolve()}")

INTER_DATA_PATH_VP = Path("vp_intermediary_data")
logger.info(f"Saving vitalperiodic specific data to {Path(INTER_DATA_PATH_VP).resolve()}")

In [ ]:
config_file_path = Path("..")  / Path("..") / Path("db") / Path("config_container.ini")
logger.info(f"Loading config for db connection from: {config_file_path.resolve()}")

#connection = DatabaseConnection(config_path=str(config_file_path.resolve()))
#engine = connection.engine
#query_schema = connection.query_schema

In [ ]:
apache_valid_results_path = INTER_DATA_PATH / Path("apache_aps_var_results_valid_mortality.csv")

df_apache_valid = pd.read_csv(filepath_or_buffer=apache_valid_results_path.resolve())
df_apache_valid = df_apache_valid.drop(columns=["Unnamed: 0", "Unnamed: 0_x", "Unnamed: 0_y"])
df_apache_valid.head(n=3)

In [ ]:
df_apache_valid.info()

# Getting the vitalperiodic data for patients with valid apache mortality prediction

In [ ]:
patient_stays_valid: list = df_apache_valid["patientunitstayid"].tolist()

Those are the patientunitstayids with a valid apache score (predictedmortality != -1)

query_valid_apache = query_schema + f"""
SELECT * 
FROM vitalperiodic
WHERE patientunitstayid in ({", ".join(map(str, patient_stays_valid))}) 
ORDER BY patientunitstayid, observationoffset
LIMIT 50000
"""

In [ ]:

if entire_dataset:
    vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_all.parquet")
else:
    vp_apache_path = Path(INTER_DATA_PATH) / Path("vital_periodic_with_apache_50000.csv")

if vp_apache_path.is_file():
    if entire_dataset:
        df_vp_apache_valid = pd.read_parquet(vp_apache_path, engine="pyarrow")
    else:
        df_vp_apache_valid = pd.read_csv(vp_apache_path, engine="pyarrow", index_col=[0])
    logger.info(f"Loading data from local file")
else:
   # df_vp_apache_valid = pd.read_sql_query(query_valid_apache, engine)
    #df_vp_apache_valid.to_csv(path_or_buf=vp_apache_path)
    df_vp_apache_valid = None
    logger.info(f"Loading data from database.")

In [ ]:
df_vp_apache_valid.info()

In [ ]:
df_vp_apache_valid.memory_usage()

In [ ]:
df_vp_apache_valid_grouped = df_vp_apache_valid.groupby("patientunitstayid")
df_vp_apache_valid_grouped.head()

In [ ]:
unique_stays = df_vp_apache_valid["patientunitstayid"].unique()
print(f"We got {len(unique_stays)} unique patientunitids with an Apache score and the vitalperiodic table in the dataset")

This number is different from the number of valid Apache Scores:

In [ ]:
print(f"Number of patientunitstayids with valid apache predictions {len(patient_stays_valid)}")
print(f"{len(patient_stays_valid) - len(unique_stays)} patientunitstayids are missing in the vitalperiodic table" )

# Determining the coverage of the variables with regard to the patientunitstayid measurements.
A measurement shall be a row of a unique patientunitstayid.

In [ ]:
vp_parameters = list()
for column in df_vp_apache_valid.columns:
    if column == "vitalperiodicid" or column == "patientunitstayid" or column == "observationoffset" or column == "":
        pass
    else:
        vp_parameters.append(column)
        

# Downcasting the vitalperiodic dtypes to be more memory efficient
Down from 17.8 GB to 9.8 GB

**only once**

In [ ]:
#df_vp_apache_valid[["vitalperiodicid","patientunitstayid","observationoffset"]] =df_vp_apache_valid[["vitalperiodicid","patientunitstayid","observationoffset"]].apply(pd.to_numeric, downcast="integer")

#df_vp_apache_valid[[*vp_parameters]] = df_vp_apache_valid[[*vp_parameters]].apply(pd.to_numeric, downcast="float")

In [ ]:
#df_vp_apache_valid.describe()

In [ ]:
vp_apache_path

In [ ]:
#df_vp_apache_valid.to_parquet(path=vp_apache_path.resolve())

parameter_sample_count_df = pd.DataFrame(data=parameter_sample_count_list_all)
parameter_coverage_ratio_df = pd.DataFrame(data=parameter_coverage_ratio_list_all)

#sns.heatmap(parameter_sample_count_df.drop(["patientunitstayid", "measurements", "observationoffset"], axis=1), annot=False)

In [ ]:
def generate_parameter_stats(df_vp_apache_valid_grouped, vp_parameters):
    measurement_id_count = dict()

    for group_id, group_df in df_vp_apache_valid_grouped:
        stay_id = group_id
        number_measurements = len(group_df)
        measurement_id_count.update({stay_id: number_measurements})
        
        # dict comprehension should make it faster 
        parameter_sample_count = {param: group_df[param].count() for param in vp_parameters}
        parameter_coverage_ratio = {param: group_df[param].count() / number_measurements for param in vp_parameters}
        parameter_missing_count = {param: number_measurements - group_df[param].count() for param in vp_parameters}
        
        parameter_sample_count.update({"patientunitstayid": stay_id})
        parameter_coverage_ratio.update({"patientunitstayid": stay_id})
        parameter_missing_count.update({"patientunitstayid": stay_id})

        yield parameter_sample_count, parameter_coverage_ratio, parameter_missing_count


In [ ]:
parameter_sample_count_list_all = []
parameter_missing_count_list_all = []
parameter_coverage_ratio_list_all = []


for sample_count, coverage_ratio, missing_count in generate_parameter_stats(df_vp_apache_valid_grouped, vp_parameters):
    parameter_sample_count_list_all.append(sample_count)
    parameter_missing_count_list_all.append(missing_count)
    parameter_coverage_ratio_list_all.append(coverage_ratio)

parameter_sample_count_df = pd.DataFrame(data=parameter_sample_count_list_all)
parameter_missing_count_df = pd.DataFrame(data=parameter_missing_count_list_all)
parameter_coverage_ratio_df = pd.DataFrame(data=parameter_coverage_ratio_list_all)


In [ ]:

parameter_sample_count_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_sample_count.csv")
parameter_missing_count_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_missing_count.csv")
parameter_coverage_count_path = Path(INTER_DATA_PATH) / Path("vital_periodic_all_coverage_count.csv")

if entire_dataset:
    parameter_sample_count_df.to_csv(path_or_buf=parameter_sample_count_path)
    parameter_missing_count_df.to_csv(path_or_buf=parameter_missing_count_path)
    parameter_coverage_ratio_df.to_csv(path_or_buf=parameter_coverage_count_path)


In [ ]:
number_stays = parameter_coverage_ratio_df.shape[0]
number_stays

In [ ]:
fig_sample, ax_sample = plt.subplots(1,1, figsize=(15,15))


sns.heatmap(parameter_sample_count_df.drop(["patientunitstayid"], axis=1), ax=ax_sample, annot=False)

ax_sample.collections[0].colorbar.set_label("Number of Samples")
plt.title(f"{number_stays} ICU stays; number of samples per variable")


filename_corr = (notebook_name + "_number_of_samples_per_variable_heatmap-" + today )
plot_file_corr_path = Path("plot_outputs") / Path(filename_corr)

if entire_dataset:
    plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

In [ ]:
fig_cov_rat, ax_cov_rat = plt.subplots(1,1, figsize=(15,15))

sns.heatmap(parameter_coverage_ratio_df.drop(["patientunitstayid"], axis=1), ax=ax_cov_rat, annot=False)

ax_cov_rat.collections[0].colorbar.set_label("Ratio")
plt.title(f"{number_stays} ICU stays; #samples/#measurements")

filename_corr = (notebook_name + "_ratio_samples_measurement_per_variable_heatmap-" + today)
plot_file_corr_path = Path("plot_outputs") / Path(filename_corr)

if entire_dataset:
    plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

In [ ]:
fig_missing, ax_missing = plt.subplots(1,1, figsize=(15,15))

sns.heatmap(parameter_missing_count_df.drop(["patientunitstayid"], axis=1), ax=ax_missing, annot=False)

ax_missing.collections[0].colorbar.set_label("Number")
plt.title(f"{number_stays} ICU stays; #missing values")

filename_corr = (notebook_name + "_number_missing_values_per_variable_heatmap-" + today)
plot_file_corr_path = Path("plot_outputs") / Path(filename_corr)

if entire_dataset:
    plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

#### Sum of missing values over the vitalperiodic parameters


In [ ]:
sum_missing_vals_df = parameter_missing_count_df.sum().to_frame()
sum_missing_vals_df.columns = ["NumMissingVals"]
sum_missing_vals_df

# Descriptive Statistics of Vitalperiodic with valid Apache predictions

In [ ]:
vp_apache_stats = df_vp_apache_valid[vp_parameters].describe()

In [ ]:
print(vp_apache_stats)


In [ ]:
vp_apache_stats_path = Path(INTER_DATA_PATH_VP) / Path(f"vital_periodic_all_stats-{today}.csv")

if entire_dataset:
    vp_apache_stats.to_csv(path_or_buf=vp_apache_stats_path, index=False)
    

# Histograms, Percentiles and Boxplots


## Histograms and determining the 0.01%, 0.10%, 99.9%, 99.99% Percentiles of the table

In [ ]:


fig_hists, axs_hists = plt.subplots(ncols=3, nrows=len(vp_parameters) // 3 + 1, figsize=(10,10))

percentiles_place = [0.0001, 0.001, 0.1, 0.9, 0.999, 0.9999]
percentiles_labels = [f"{percentile:.2%}" for percentile in percentiles_place]

percentiles_all = {}

for idx, param in enumerate(vp_parameters):
    sns.histplot(df_vp_apache_valid[param], ax=axs_hists.flat[idx], kde=False)
    plt.suptitle(f"Histogram for {param}")
    
    value_at_percentile = df_vp_apache_valid[param].quantile(percentiles_place)
    percentiles_all[param] = value_at_percentile

    y_axis_limits = axs_hists.flat[idx].get_ylim()
    percentile_colors = palettable.tableau.ColorBlind_10.mpl_colors
    
    for j, perc_val in enumerate(value_at_percentile):
        axs_hists.flat[idx].vlines(perc_val, linestyle="--", colors=percentile_colors[j], ymin=y_axis_limits[0], ymax=y_axis_limits[1], label=percentiles_labels[j])

param_percentiles_df = pd.DataFrame.from_dict(percentiles_all)

leg = fig_hists.legend(
    [*percentiles_labels],
    title="Percentiles",
    loc="upper center",
    bbox_to_anchor=(0.50, 0),
    frameon=False,
    fancybox=True,
    shadow=True,
    ncol=4,
    #prop=font_consistent,
)


fig_hists.suptitle("Histograms Vitalperiodic")
fig_hists.tight_layout()


filename_corr = "histograms_vitalperiodic-" + today

if entire_dataset:
    plot_file_corr_path = Path("plot_outputs") / Path(filename_corr)
    plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

## Boxplots for multiple quantiles

In [ ]:
fig_hists, axs_hists = plt.subplots(ncols=3, nrows=len(vp_parameters) // 3 + 1, figsize=(10,10))


for idx, param in enumerate(vp_parameters):
    sns.boxenplot(df_vp_apache_valid[param], ax=axs_hists.flat[idx], fill=False)
    #plt.suptitle(f"Boxplot for {param}")
    

fig_hists.suptitle("Boxplots Vitalperiodic")
fig_hists.tight_layout()

filename_corr = "boxplots_vitalperiodic-" + today

if entire_dataset:
    plot_file_corr_path = Path("plot_outputs") / Path(filename_corr)
    plt.savefig(str(plot_file_corr_path) + ".png", dpi=300, bbox_inches="tight")

#### Does removing the 0.01% percentiles make sense? Or rather 0.001%
**0.01% could remove valuable outliers, especially for ranges that are not suffering from "unphysical" values**

#### Cutoff values according to the calcualted percentiles:

In [ ]:
param_percentiles_df

In [ ]:
filename_vp_percentiles = f"vital_periodic_percentiles-{today}.csv"
path_vp_percentiles = Path(INTER_DATA_PATH_VP) / Path(filename_vp_percentiles)


param_percentiles_df.to_csv(path_or_buf=path_vp_percentiles, index=False)

#### Removing the data that is not withing the 0.001% < x < 99.999% range 

In [ ]:
percentile_removal_bounds = [0.0001, 0.9999]
#percentile_removal_bounds = [0.1, 0.9]
aux_columns = ["vitalperiodicid", "patientunitstayid", "observationoffset"]

df_vp_apache_valid_filtered = pd.DataFrame(index=df_vp_apache_valid.index)
df_vp_apache_valid_filtered[[*aux_columns]] = df_vp_apache_valid[[*aux_columns]] 

for col in param_percentiles_df:
   lower_bound = param_percentiles_df[col].loc[percentile_removal_bounds[0]]
   upper_bound = param_percentiles_df[col].loc[percentile_removal_bounds[1]]
   
   df_vp_apache_valid_filtered[col] = df_vp_apache_valid[col].loc[(df_vp_apache_valid[col] > lower_bound) & (df_vp_apache_valid[col] < upper_bound)]

In [ ]:
df_vp_apache_valid_filtered.describe()

In [ ]:
filename_vp_filtered = "vital_periodic_outlier_filtered.parquet"
path_vp_filtered = Path(INTER_DATA_PATH_VP) / Path(filename_vp_filtered)

if entire_dataset:
    df_vp_apache_valid_filtered.to_parquet(path=path_vp_filtered, engine="pyarrow", index=False)

In [ ]:
df_vp_apache_valid.describe()

Plenty of values already got removed by filtering via the 0.001 & 0.9999 quantiles as we can see by comparing both datatframe descriptions.
But some of the unrealistic samples are still in there.

However, sao2 is "over-filtered", the max value of this is now 9.900000e+01 even though we just wanted to remove values like 2.390000e+02.

**Overfiltered values (0.001 % & 99.999%):**
- sao2: 2.390000e+02 -> 9.900000e+01
- heartrate  3.000000e+02 -> 1.740000e+02 (maximum heartrate of human is 220 ) # Citation für bound 


**Still outliers (0.001% & 99.999%):**
- temperature: 1.028500e+02 (100 degree body temperature?)
- respiration: 6.301200e+04
- etco2: (siehe W03-2024)

Genaues Vorgehen nochmal mit Kai besprechen, ich hab bisschen Bedenken was den Quantilen Ansatz betrifft


In [ ]:
vp_apache_stats_filtered_path = Path(INTER_DATA_PATH_VP) / Path(f"vital_periodic_outlier_filtered_stats-{today}.csv")

if entire_dataset:
    vp_apache_stats.to_csv(path_or_buf=vp_apache_stats_filtered_path, index=False)